In [1]:
# 评估Aliyun
from official_api.aliyun import face_compare
import torch
from tqdm import tqdm
import os
import numpy as np

ths=[61,69,75]
result = {}
model_name="ArcFace"
with torch.no_grad():
    # 设置对抗样本目录路径 每月免费10000次调用，测三个刚好9000次，换个免费的继续测
    adv_samples_dirs = [
        f"data/FGSM_{model_name}_lfw_eps6_tpert4.4",
        f"data/MIM_{model_name}_lfw_eps6_tpert4.4",
        f"data/CW_{model_name}_lfw_eps16_tpert1",
        f"data/AT3D_{model_name}_eye_nose_lfw_eps5_tpert13.5",
        f"data/SiblingAttack_{model_name}_lfw_eps0.15_tpert10.7",
        f"data/AdvFace_lfw_eps8_tpert5.7",
        f"data/AdvMakeUP_lfw_tpert5.2",
        f"data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6",
    ]
    for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
        print("-----------------start white evaluate target method {0}-------------------\n".format(adv_samples_dir))
        save_path=f'apiresult/aliyun/{adv_samples_dir[5:]}.npy'
        if not os.path.exists(save_path):
            result[adv_samples_dir]= np.empty((0, 3), float)
            # 列出目录中的所有文件和文件夹
            all_files_and_dirs = os.listdir(adv_samples_dir)
            # 过滤出所有子文件夹
            subdirectories = [d for d in all_files_and_dirs if os.path.isdir(os.path.join(adv_samples_dir, d))]
            for adv_pair in tqdm(subdirectories):
                base_dir = os.path.join(adv_samples_dir,adv_pair)
                adv_path = base_dir+"/adv.png"
                source_path = base_dir+"/source.png"
                target_path = base_dir+"/target.png"
                base_res = face_compare(face1_path=source_path, face2_path=target_path)
                FSS_res = face_compare(face1_path=adv_path, face2_path=source_path)
                FTS_res = face_compare(face1_path=adv_path, face2_path=target_path)
                if base_res is not None and FSS_res is not None and FTS_res is not None:
                    result[adv_samples_dir] = np.vstack([result[adv_samples_dir], [base_res,FSS_res,FTS_res]])
            print("aliyun在"+adv_samples_dir+"上失败了："+str(1000-len(result[adv_samples_dir])))
            np.save(save_path, np.array(result[adv_samples_dir]))
        else:
            result[adv_samples_dir] = np.load(save_path)
        num = len(result[adv_samples_dir])
        print("有效数据个数 ",num)
        for th in ths:
            print(th, np.sum(result[adv_samples_dir][:,0] > th)/num, np.sum(result[adv_samples_dir][:,2] > th)/num, np.sum((result[adv_samples_dir][:,1] > th) & (result[adv_samples_dir][:,2] > th))/num)

-----------------start white evaluate target method data/FGSM_ArcFace_lfw_eps6_tpert4.4-------------------

有效数据个数  999
61 0.0 0.25225225225225223 0.25225225225225223
69 0.0 0.05005005005005005 0.05005005005005005
75 0.0 0.003003003003003003 0.003003003003003003
-----------------start white evaluate target method data/MIM_ArcFace_lfw_eps6_tpert4.4-------------------

有效数据个数  1000
61 0.0 0.723 0.719
69 0.0 0.367 0.343
75 0.0 0.076 0.049
-----------------start white evaluate target method data/CW_ArcFace_lfw_eps16_tpert1-------------------

有效数据个数  1000
61 0.0 0.002 0.002
69 0.0 0.0 0.0
75 0.0 0.0 0.0
-----------------start white evaluate target method data/AT3D_ArcFace_eye_nose_lfw_eps5_tpert13.5-------------------

有效数据个数  1000
61 0.001 0.48 0.389
69 0.0 0.112 0.057
75 0.0 0.003 0.0
-----------------start white evaluate target method data/SiblingAttack_ArcFace_lfw_eps0.15_tpert10.7-------------------

有效数据个数  998
61 0.001002004008016032 0.9468937875751503 0.4749498997995992
69 0.0 0.81

In [2]:
print("aliyun 0.01%FAR ASR1:")
for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
    num = len(result[adv_samples_dir])
    print(f"{np.sum(result[adv_samples_dir][:,2] > ths[1])/num*100:.1f}")
print("aliyun 0.01%FAR ASR2:")
for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
    num = len(result[adv_samples_dir])
    print(f"{np.sum((result[adv_samples_dir][:, 1] > ths[1]) & (result[adv_samples_dir][:, 2] > ths[1])) / num*100:.1f}")

aliyun 0.01%FAR ASR1:
5.0
36.7
0.0
11.2
82.0
21.6
0.0
79.3
aliyun 0.01%FAR ASR2:
5.0
34.3
0.0
5.7
16.5
11.9
0.0
62.1


In [3]:
print("aliyun 0.1%FAR ASR1:")
for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
    num = len(result[adv_samples_dir])
    print(f"{np.sum(result[adv_samples_dir][:,2] > ths[0])/num*100:.1f}")
print("aliyun 0.1%FAR ASR2:")
for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
    num = len(result[adv_samples_dir])
    print(f"{np.sum((result[adv_samples_dir][:, 1] > ths[0]) & (result[adv_samples_dir][:, 2] > ths[0])) / num*100:.1f}")

aliyun 0.1%FAR ASR1:
25.2
72.3
0.2
48.0
94.7
45.6
1.2
93.5
aliyun 0.1%FAR ASR2:
25.2
71.9
0.2
38.9
47.5
38.0
1.2
89.6
